In [2]:
import os
import sys
from pathlib import Path

cur_dir = os.getcwd()

# Resolve ARIES root robustly when this notebook is moved (e.g., under tests/)
cwd = Path(cur_dir).resolve()
aries_root = None
for candidate in [cwd] + list(cwd.parents):
    maybe = candidate / "tools" / "ARIES"
    if maybe.exists():
        aries_root = maybe
        break

if aries_root is None:
    raise RuntimeError("Cannot locate tools/ARIES from current working directory.")

aries_path = str(aries_root)
sys.path.append(aries_path)
from frontend import *
from IPython import get_ipython

In [3]:
# GEMM0: C[i0, j0] += A[i0, k0] * B[k0, j0] (64*64*128)
I, J, K = 256, 256, 256
TI, TJ, TK = 32, 32, 32
GRID_I0, GRID_J0, GRID_K0 = (I // TI, J // TJ, K // TK)

In [4]:
@task_kernel(external_path="aie1/adf/kernel_mm/aie_fp32_v0", para = [TI, TJ, TK])
def kernel_gemm(TileA: float32[TI, TK], 
                TileB: float32[TK, TJ], 
                TileC: float32[TI, TJ]):
    for i0 in range(0, TI):
        for j0 in range(0, TJ):
            TileC[i0, j0] = float32(0)
            for k0 in range(0, TK):
                TileC[i0, j0] += TileA[i0, k0] * TileB[k0, j0]

In [5]:
@task_tile()
def gemm(A: float32[-1, -1], B: float32[-1, -1], 
         C: float32[-1, -1], GRID_I, GRID_J, GRID_K):
    for i in range(GRID_I):
        for j in range(GRID_J):
            for k in range(GRID_K):
                # Compute tile slices for multiple dimensions
                ti = aries.arange(i*TI, (i+1)*TI)  # I tile range
                tj = aries.arange(j*TJ, (j+1)*TJ)  # J tile range
                tk = aries.arange(k*TK, (k+1)*TK)  # K tile range

                L1_A = aries.buffer((TI, TK), "float32")
                L1_B = aries.buffer((TK, TJ), "float32")
                L1_C = aries.buffer((TI, TJ), "float32")

                L1_A = aries.load(A, (ti, tk))
                L1_B = aries.load(B, (tk, tj))
                kernel_gemm(L1_A, L1_B, L1_C)
                aries.accstore(L1_C, C, (ti, tj))

In [6]:
@task_top()
def top(A: float32[I, K], B: float32[K, J], C: float32[I, J]):
    cast_A = aries.cast(A, (-1, -1)) # This is for lowering
    cast_B = aries.cast(B, (-1, -1))
    cast_C = aries.cast(C, (-1, -1))
    gemm_task = gemm(cast_A, cast_B, cast_C, GRID_I0, GRID_J0, GRID_K0)
    return gemm_task

In [7]:
# Build source code for sch.build() from live task objects (robust to re-run order)
import inspect
import textwrap

const_src = "\n".join([
    f"I, J, K = {I}, {J}, {K}",
    f"TI, TJ, TK = {TI}, {TJ}, {TK}",
    f"GRID_I0, GRID_J0, GRID_K0 = ({GRID_I0}, {GRID_J0}, {GRID_K0})",
])

def _unwrap_task_fn(task_obj):
    return task_obj.func if hasattr(task_obj, "func") else task_obj

kernel_src = textwrap.dedent(inspect.getsource(_unwrap_task_fn(kernel_gemm)))
tile_src = textwrap.dedent(inspect.getsource(_unwrap_task_fn(gemm)))
top_src = textwrap.dedent(inspect.getsource(_unwrap_task_fn(top)))

all_code = "\n\n".join([const_src, kernel_src, tile_src, top_src])

In [8]:
# Initialize the buffers
np.random.seed(0)
A = np.random.rand(I, K).astype(np.float32)
B = np.random.rand(K, J).astype(np.float32)
C = np.zeros((I, J)).astype(np.float32)

# Execute on CPU
gemm_task = top(A, B, C)

# Golden file generation
golden_C = np.matmul(A, B)

# Compare the program with golden file
print(np.allclose(C, golden_C))

# # Generate files for on-board test
aries.gen_sim([A, B, golden_C])

True


In [9]:
# Specify primitives to optimize hardware design
sch = Schedule(gemm_task)

############# Primitives #############
sch.parallel(gemm_task, [1, 1, 1]) # AIE Array Parallelism
sch.l2buffer(gemm_task, [2, 2, 2]) # L2 buffer data reuse
sch.bufsel(gemm_task, [1, 1, 0]) # Select the type of buffer of A, B, C, 1:BRAM; 0:URAM
sch.aieVector(gemm_task, 1) # Conservative vectorization to reduce backend scheduling pressure
######################################

sch.to("VCK190")

In [10]:
# Set the project dir and template dir
prj_dir= cur_dir + '/project_gemm_dyn'
temp_dir= aries_path + '/templates'
# Generate Initial MLIR and ARIES Opts
sch.build(all_code, prj_dir, temp_dir)

Created directory: /home/byao/Desktop/uni-comb-accel/tests/project_gemm_dyn/project
Created directory: /home/byao/Desktop/uni-comb-accel/tests/project_gemm_dyn/project/aie
Created directory: /home/byao/Desktop/uni-comb-accel/tests/project_gemm_dyn/project/kernel
Created directory: /home/byao/Desktop/uni-comb-accel/tests/project_gemm_dyn/project/host
... wrote /home/byao/Desktop/uni-comb-accel/tests/project_gemm_dyn/Makefile
... wrote /home/byao/Desktop/uni-comb-accel/tests/project_gemm_dyn/project/Makefile
... wrote /home/byao/Desktop/uni-comb-accel/tests/project_gemm_dyn/project/aie/kernel_gemm.cc
... wrote /home/byao/Desktop/uni-comb-accel/tests/project_gemm_dyn/project/aie/kernel_gemm0.cc


### End-to-end correctness verification: direct compute + sw_emu + hw_emu

This section runs three checks:
1. Direct compute check on CPU (`C` vs `golden_C`).
2. `sw_emu` build + run.
3. `hw_emu` build + run.

Both emulation flows parse host output for `TEST PASSED` and write full logs under `project_gemm_dyn/project/` for debugging.

In [11]:
# 1) Direct compute correctness check (CPU path)
verification = {}

direct_max_abs = float(np.max(np.abs(C - golden_C)))
direct_ok = bool(np.allclose(C, golden_C, rtol=1e-4, atol=1e-4))

verification["direct_cpu"] = {
    "ran": True,
    "passed": direct_ok,
    "max_abs_err": direct_max_abs,
}

print("[direct_cpu]", "PASS" if direct_ok else "FAIL", f"max_abs_err={direct_max_abs:.6e}")

[direct_cpu] PASS max_abs_err=6.484985e-05


In [12]:
# 2) Helpers for software sim (aiesim) / hw_emu execution and log parsing
import re
import subprocess
from pathlib import Path

def _sim_to_vtxt(src: Path, dst: Path, width: int = 4):
    tokens = src.read_text().split()
    with dst.open("w") as f:
        for i in range(0, len(tokens), width):
            f.write(" ".join(tokens[i:i + width]) + "\n")

def run_emu_target(target: str, edge_common_sw_path: str, *, clean_first: bool = False,
                   xilinx_root: str = "/tools/Xilinx/2025.1", timeout_sec: int = 7200):
    assert target in ("sw_emu", "hw_emu"), f"Unsupported target: {target}"
    top_dir = Path(prj_dir)
    project_dir = top_dir / "project"
    log_file = project_dir / f"run_{target}.log"

    vitis_settings = f"{xilinx_root}/Vitis/settings64.sh"
    if not Path(vitis_settings).exists():
        raise FileNotFoundError(f"Vitis settings script not found: {vitis_settings}")

    xrt_candidates = [
        "/opt/xilinx/xrt/setup.sh",
        f"{xilinx_root}/xrt/setup.sh",
        f"{xilinx_root}/Vitis/xrt/setup.sh",
    ]
    xrt_setup = next((p for p in xrt_candidates if Path(p).exists()), None)

    aries_bin = Path(aries_path) / "build" / "bin"

    shell_lines = [
        "set -e",
        f"source '{vitis_settings}'",
    ]
    if xrt_setup:
        shell_lines.append(f"source '{xrt_setup}'")
    else:
        shell_lines.append("echo '[warn] XRT setup.sh not found; continuing with current env.'")
    shell_lines.append(f"export PATH='{aries_bin}':$PATH")

    if clean_first:
        shell_lines.append(f"cd '{project_dir}'")
        shell_lines.append("make clean || true")

    # Generate ARIES-translated project artifacts (adf_graph.cpp, kernels, host, etc.)
    shell_lines.append(f"cd '{top_dir}'")
    shell_lines.append("make all")

    # Vitis 2025.1 no longer supports sw_emu for v++; use aiesim as software simulation.
    if target == "sw_emu":
        # aiesim in this template expects ./data/v0.txt and ./data/v1.txt
        data_dir = project_dir / "data"
        data_dir.mkdir(exist_ok=True)
        data0 = project_dir / "data0.sim"
        data1 = project_dir / "data1.sim"
        if data0.exists():
            _sim_to_vtxt(data0, data_dir / "v0.txt", width=4)
        if data1.exists():
            _sim_to_vtxt(data1, data_dir / "v1.txt", width=4)

        shell_lines.append(f"cd '{project_dir}'")
        shell_lines.append("make aiesim TARGET=sw_emu")
    else:
        shell_lines.append(f"cd '{project_dir}'")
        shell_lines.append(f"make all TARGET={target} EDGE_COMMON_SW_PATH='{edge_common_sw_path}'")
        shell_lines.append(f"make run_emu TARGET={target} EDGE_COMMON_SW_PATH='{edge_common_sw_path}'")

    cmd = "\n".join(shell_lines)

    try:
        proc = subprocess.run(
            ["bash", "-lc", cmd],
            text=True,
            capture_output=True,
            timeout=timeout_sec,
            check=False,
        )
        output = (proc.stdout or "") + "\n" + (proc.stderr or "")
        log_file.write_text(output)

        if target == "sw_emu":
            passed = (
                "COMPLETE: aiesim success." in output
                or "Simulation completed successfully" in output
            )
        else:
            passed = (
                "TEST PASSED" in output
                or (re.search(r"\bPASSED\b", output) is not None and "TEST FAILED" not in output)
            )

        verification[target] = {
            "ran": True,
            "passed": bool(passed and proc.returncode == 0),
            "return_code": int(proc.returncode),
            "log": str(log_file),
            "xrt_setup": xrt_setup,
            "mode": "aiesim" if target == "sw_emu" else "hw_emu",
        }

    except subprocess.TimeoutExpired as e:
        output = (e.stdout or "") + "\n" + (e.stderr or "")
        log_file.write_text(output)
        verification[target] = {
            "ran": True,
            "passed": False,
            "return_code": None,
            "timeout_sec": timeout_sec,
            "log": str(log_file),
            "xrt_setup": xrt_setup,
            "mode": "aiesim" if target == "sw_emu" else "hw_emu",
        }

    status = verification[target]
    print(f"[{target}] {'PASS' if status['passed'] else 'FAIL'} | mode={status['mode']} | log={status['log']}")
    if status.get("xrt_setup"):
        print(f"[{target}] XRT setup: {status['xrt_setup']}")
    else:
        print(f"[{target}] XRT setup: not found (continued)")
    return status

In [13]:
# 3) Run software simulation and hw_emu
# Point to the folder that contains xilinx-versal-common-v2025.1
EDGE_COMMON_SW_PATH = "/home/arclab/"

# RUN_SW_EMU uses aiesim mode on Vitis 2025.1
RUN_SW_EMU = True
RUN_HW_EMU = True

if RUN_SW_EMU:
    run_emu_target("sw_emu", EDGE_COMMON_SW_PATH, clean_first=False, timeout_sec=7200)

if RUN_HW_EMU:
    run_emu_target("hw_emu", EDGE_COMMON_SW_PATH, clean_first=False, timeout_sec=14400)

[sw_emu] PASS | mode=aiesim | log=/home/byao/Desktop/uni-comb-accel/tests/project_gemm_dyn/project/run_sw_emu.log
[sw_emu] XRT setup: not found (continued)
[hw_emu] FAIL | mode=hw_emu | log=/home/byao/Desktop/uni-comb-accel/tests/project_gemm_dyn/project/run_hw_emu.log
[hw_emu] XRT setup: not found (continued)


In [15]:
# 4) Final summary
from pathlib import Path

required_checks = ["direct_cpu", "sw_emu", "hw_emu"]

print("\nVerification summary:")
for key in required_checks:
    info = verification.get(key, {"ran": False, "passed": False})
    ran = info.get("ran", False)
    passed = info.get("passed", False)
    extra = []
    if key == "direct_cpu" and "max_abs_err" in info:
        extra.append(f"max_abs_err={info['max_abs_err']:.6e}")
    if "return_code" in info:
        extra.append(f"return_code={info['return_code']}")
    if "timeout_sec" in info:
        extra.append(f"timeout={info['timeout_sec']}s")
    if "log" in info:
        extra.append(f"log={info['log']}")

        # Surface known backend/toolchain failures explicitly
        log_path = Path(info["log"])
        if log_path.exists():
            log_text = log_path.read_text(errors="ignore")
            if "unsatisfied data/anti-dependency" in log_text:
                extra.append("known_hw_backend_issue=chess-backend data/anti-dependency")
            if "software emulation target 'sw_emu' is no longer supported" in log_text:
                extra.append("known_sw_emu_target_deprecated=true")

    extra_str = (" | " + " | ".join(extra)) if extra else ""
    print(f"- {key}: {'PASS' if (ran and passed) else 'FAIL/NOT-RUN'}{extra_str}")

all_passed = all(verification.get(k, {}).get("passed", False) for k in required_checks)
print("\nOverall verdict:", "PASS" if all_passed else "FAIL")


Verification summary:
- direct_cpu: PASS | max_abs_err=6.484985e-05
- sw_emu: PASS | return_code=0 | log=/home/byao/Desktop/uni-comb-accel/tests/project_gemm_dyn/project/run_sw_emu.log
- hw_emu: FAIL/NOT-RUN | return_code=2 | log=/home/byao/Desktop/uni-comb-accel/tests/project_gemm_dyn/project/run_hw_emu.log | known_hw_backend_issue=chess-backend data/anti-dependency

Overall verdict: FAIL
